<a href="https://colab.research.google.com/github/FURK4NGG/cv-model-training-pt/blob/main/Universal_YOLO_Hailo_Training_EN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Universal YOLO → ONNX → Hailo HEF Training Notebook

This notebook is designed to be reused across different object-detection projects.

When starting a new project, you normally only need to edit **Section 1: Single User Configuration Cell**. The project name, Drive paths, dataset ZIP name, YOLO model, class list, training settings, test thresholds, ONNX settings, and Hailo target are all managed from that cell.

This notebook:

1. Uses a dataset ZIP stored in Google Drive.
2. Validates the dataset structure and YOLO annotations.
3. Supports new training, exact resume, and additional-epoch extend modes.
4. Produces metrics, a confusion matrix, and prediction visualizations on a separate test split.
5. Performs visual testing on new field images.
6. Produces `best.pt` and fixed-shape `best.onnx` files.
7. Attempts direct HEF export when a compatible Hailo DFC is available.
8. Creates a Hailo Docker compilation package when direct export is not available.

> Enable a GPU: **Runtime → Change runtime type → T4 GPU**, or select a more powerful GPU.

> Hailo note: `BASE_MODEL` and `HAILO_MODEL_ZOO_NETWORK` must represent the same architecture. For example, use `yolov11l` for `yolo11l.pt`, and `yolov11n` for `yolo11n.pt`.


## Before You Begin: Selecting a GPU

Select a GPU in Colab using the following steps:

1. Open the notebook.
2. Select **Runtime** from the top menu.
3. Select **Change runtime type**.
4. Set **Hardware accelerator** to `GPU`.
5. If available, choose an option under **GPU class / Runtime shape**:

   ```text
   T4 GPU
   L4 GPU
   A100 GPU
   ```

L4 and A100 GPUs generally require paid compute units and current hardware availability. After connecting, check the `nvidia-smi` output in the first code cell to confirm which GPU was actually assigned.

## Sources for Bounding-Box-Labeled Detection Datasets

The following sources provide detection datasets containing object bounding boxes:

- [Open Images V7](https://docs.ultralytics.com/datasets/detect/open-images-v7): a strong starting source for general classes such as `person`, `dog`, `cattle`, `bear`, and `deer`.
- [LILA BC Camera Traps](https://lila.science/datasets/): real camera-trap, nighttime/IR, and empty-forest images. Not every sub-dataset contains bounding boxes, so select datasets whose descriptions explicitly mention **bounding boxes**.
- [ENA24 Detection](https://lila.science/datasets/ena24detection/): a LILA dataset containing camera-trap images and bounding boxes.
- [Roboflow Universe](https://universe.roboflow.com/): search using terms such as `class:bear`, `class:wild boar`, `class:deer`, `class:wolf`, `class:cattle`, `class:person`, and `class:dog`, then export in `YOLOv8/YOLO11` format.
- [Kaggle Datasets](https://www.kaggle.com/datasets): use only datasets explicitly described as `object detection`, `YOLO`, or `bounding box` datasets.

Review every bounding box and class in community datasets. `pig` and `wild boar`, dog and wolf, or an American black bear and a brown bear found in Turkey are not necessarily equivalent targets. Public datasets cannot fully replace real Camera Module 3 NoIR + 850 nm IR images from the deployment environment.

## Roboflow: Exporting Only One Class from a Multi-Class Dataset

The `Versions` page does not contain a direct class-selection checkbox. Class selection is performed in the **Preprocessing** section while creating a new version:

1. If the Universe dataset belongs to someone else, first copy it to your workspace using **Download Dataset → Fork/Import to Workspace**.
2. In your own project, select **Versions** from the left menu.
3. Select **Generate New Version**.
4. On the new-version screen, select **Preprocessing → Add Preprocessing Step → Modify Classes**.
5. Keep the required class and set all other classes to **Omit**.
6. Add **Filter Null**. This removes images that become unlabeled after the other classes are removed.
7. Generate the version with **Generate**, then download it using **Download Dataset → YOLOv11 → Download ZIP**.

Example: if the source contains `people`, `car`, and `house`:

| Source class | Modify Classes action |
|---|---|
| `people` | Remap/Override as `person` |
| `car` | Omit |
| `house` | Omit |

The result is a new version in which only person boxes remain in images containing `people`, and the class is named `person`. If an image contains both a person and a car, the image remains but the car box is removed. Images containing only cars or houses are removed by `Filter Null`.

### If `Modify Classes` Does Not Appear on the New-Version Screen

First verify that the project has actually been forked into your own workspace; you cannot modify the original Universe project in place. If the option still does not appear, use the following method only in the **forked working copy**:

1. Open **Classes & Tags → Modify Classes**.
2. Enter the new name in the required class's **Override** field.
3. Delete or remove unwanted classes.
4. Then apply **Filter Null** under **Versions → Generate New Version**.

> This second method changes the project's classes and may be difficult to undo. Use it only on a forked working copy, not on the source dataset. The `Preprocessing → Modify Classes` method, which affects only the new version, is safer.

### Standardizing Unusual Class Names

Map classes that represent the same target to one standard English name:

```text
identified person → person
people          → person
human           → person
pedestrian      → person   # only if the box actually represents a pedestrian/person
```

Remapping or overriding multiple source classes to the same `person` name merges them. However, do not merge semantically different classes such as `rider`, `mannequin`, or `statue` into `person` without inspecting them. For compatibility, use English ASCII letters, numbers, and `-` when necessary; avoid spaces and Turkish characters in class names.

### A Roboflow Class Name and a YOLO Class ID Are Not the Same Thing

Renaming a class in Roboflow does not directly select the numeric value stored in the `.txt` file. In a YOLO export, the actual ID order is determined by the `names` list in the downloaded `data.yaml` file. A single-class export usually looks like this:

```yaml
names:
  0: person
```

When this ZIP is merged into the following seven-class main dataset, the ID must be converted to the main class order. The fixed order in this example project is:

```text
0 bear
1 boar
2 deer
3 wolf
4 cow
5 person
6 dog
```

Therefore, `0 person` in the single-class Roboflow export must become **`5 person`** in the main dataset. Do not use `1` for `person`; in this project, `1` represents `boar`. When editing annotations, change the `class_id`, which is the first space-separated field, not merely the first character of the line. Do not modify the coordinates.

## Recommended Data Quantities

These targets refer to genuinely distinct images. Hundreds of similar frames extracted from the same video do not count as independent data. A deviation of approximately 20% is acceptable.

| Class | Train/day | Val/day | Test/day | Train/night | Val/night | Test/night | Total |
|---|---:|---:|---:|---:|---:|---:|---:|
| Bear `bear` | 576 | 72 | 72 | 384 | 48 | 48 | **1,200** |
| Wild boar `boar` | 768 | 96 | 96 | 512 | 64 | 64 | **1,600** |
| Deer `deer` | 768 | 96 | 96 | 512 | 64 | 64 | **1,600** |
| Wolf `wolf` | 864 | 108 | 108 | 576 | 72 | 72 | **1,800** |
| Cow `cow` | 576 | 72 | 72 | 384 | 48 | 48 | **1,200** |
| Person `person` | 720 | 90 | 90 | 480 | 60 | 60 | **1,500** |
| Dog `dog` | 864 | 108 | 108 | 576 | 72 | 72 | **1,800** |
| Negative/empty | 880 | 110 | 110 | 880 | 110 | 110 | **2,200** |
| **Total** | **6,016** | **752** | **752** | **4,304** | **538** | **538** | **12,900** |

The `night` columns should represent genuine IR/nighttime images whenever possible, not daytime photographs that were later converted to grayscale.

## Images That Do Not Contain Animals

These are **negative images**, and they are necessary.

Example:

```text
images/train/empty_forest_001.jpg
labels/train/empty_forest_001.txt
```

The `empty_forest_001.txt` file must exist but remain completely empty. Never write any of the following:

```text
background
-1
0 0 0 0 0
```

Use the same system for validation and test:

```text
images/val/empty_night_001.jpg
labels/val/empty_night_001.txt       # empty file

images/test/empty_night_002.jpg
labels/test/empty_night_002.txt      # empty file
```

The notebook validation process searches for a matching `.txt` file for every image, so an empty annotation file must still be created.

### What Should Negative Images Contain?

Do not use only completely empty forests. Include difficult images that may confuse the model:

- Empty forests at night.
- Leaves reflecting infrared illumination.
- Tree trunks and logs.
- Rocks and bushes.
- Rain, fog, and snow.
- Spider webs.
- Insects close to the camera.
- Branches moving in the wind.
- Eye-like reflections.
- Vehicle headlights.
- Animal-like shadows.
- Non-target animals such as cats, birds, horses, and sheep.

However, if an image contains `dog`, `person`, or any other configured target class, do not leave the annotation empty; annotate that object with a bounding box.

### How Many Negative Images?

Use negative images equal to approximately 15–25% of the number of positive images. A recommended total is **2,000–2,500** negative images:

```text
train: 1,600–2,000
val:     200–250
test:    200–250
```

At least half of the negative images should be genuine nighttime/IR images.


## 0. Prepare the Dataset Once

Create the project directory and dataset ZIP file that you will use in Google Drive. Their exact names are defined in the next user-configuration cell.

Example:

```text
MyDrive/MyProject/my_dataset.zip
```

The ZIP must contain the following structure:

```text
images/
  train/
  val/
  test/
labels/
  train/
  val/
  test/
data.yaml
```

Every image must have a YOLO `.txt` annotation with the same filename stem. Empty `.txt` files can be used for negative images that contain no target object.

Detection annotation row:

```text
class_id x_center y_center width height
```

Coordinates must be within the `0–1` range. Do not place similar frames from the same video or camera-trap event in different splits; keep every frame from one event in the same split.


In [ ]:
# GPU and runtime environment
import os, platform, subprocess, sys

print("Python:", sys.version)
print("Platform:", platform.platform())
subprocess.run(["nvidia-smi"], check=False)

In [ ]:
# Mount Google Drive. Datasets, checkpoints, and results remain persistent here.
from google.colab import drive
# Force-remount Google Drive at the standard Colab mount point.
drive.mount('/content/drive', force_remount=True)

## 1. Single User Configuration Cell

When starting a new project, you normally only need to modify the following cell.

Check these settings carefully:

- `PROJECT_NAME`: project directory used in Google Drive.
- `PROJECT_SLUG`: short name used in filenames and training-run names.
- `DATASET_ZIP_NAME`: name of the dataset ZIP uploaded to Drive.
- `BASE_MODEL`: Ultralytics model to train, such as `yolo11n.pt`, `yolo11m.pt`, or `yolo11l.pt`.
- `CLASS_NAMES`: class list in exactly the same order as the label IDs and `data.yaml`.
- `HAILO_MODEL_ZOO_NETWORK`: Hailo Model Zoo network name matching the ONNX architecture.
- `HAILO_HW_ARCH`: target Hailo hardware architecture.

`TRAIN_MODE` options:

- `new`: starts a new training run using the selected `BASE_MODEL`.
- `resume`: exactly resumes the interrupted run from the newest `last.pt`, including optimizer state.
- `extend`: starts an additional training stage from the newest `best.pt` weights.

> Do not share the same Drive project directory between unrelated models. Checkpoint discovery selects the newest `best.pt` and `last.pt` under the project directory. Use a different `PROJECT_NAME` for every independent project.


In [ ]:
from pathlib import Path

# ============================================================
# USER CONFIGURATION — NORMALLY EDIT ONLY THIS SECTION FOR A NEW PROJECT
# ============================================================

# Project and filenames
PROJECT_NAME = 'WildlifeYOLO'             # Project directory in Google Drive
PROJECT_SLUG = 'wildlife'                 # Short name without spaces; used in run/export names
DRIVE_BASE = Path('/content/drive/MyDrive')
DATASET_ZIP_NAME = 'wildlife_dataset.zip'

# Model and dataset
BASE_MODEL = 'yolo11l.pt'                 # Examples: yolo11n.pt, yolo11m.pt, yolo11l.pt
IMAGE_SIZE = 640
CLASS_NAMES = ['bear', 'boar', 'deer', 'wolf', 'cow', 'person', 'dog']

# Training
TRAIN_MODE = 'new'                        # 'new', 'resume', 'extend' or 'finetune'
NEW_EPOCHS = 150
EXTRA_EPOCHS = 40
FINE_TUNE_EPOCHS = 30                 # Dataset güncellendikten sonra best.pt üzerinden yeni eğitim
FINE_TUNE_LR0 = 0.001                      # Dataset değişikliğinde kullanılacak düşük başlangıç learning rate
BATCH = -1                                # -1: Ultralytics selects automatically according to GPU memory
PATIENCE = 30
WORKERS = 2
SEED = 42
CLOSE_MOSAIC = 15

# Test and prediction thresholds
TEST_BATCH = 8
TEST_CONF = 0.001
TEST_IOU = 0.60
PREDICT_CONF = 0.20
PREDICT_IOU = 0.60
VISUAL_SAMPLE_COUNT = 12
MAX_PREDICTION_PREVIEWS = 20

# ONNX export
ONNX_OPSET = 11
ONNX_SIMPLIFY = True
ONNX_DYNAMIC = False
ONNX_BATCH = 1

# Hailo export / Docker compilation settings
HAILO_MODEL_ZOO_NETWORK = 'yolov11l'      # Must match the BASE_MODEL architecture
HAILO_EXPORT_NAME = 'hailo8'              # Ultralytics Hailo export target name
HAILO_HW_ARCH = 'hailo8'                  # Value passed to hailomz --hw-arch
HAILO_TARGET_DESCRIPTION = 'Hailo-8 26 TOPS'
HAILO_CALIBRATION_COUNT = 2048
HAILO_CALIBRATION_FRACTION = 1.0
HAILO_CONF = 0.10
HAILO_IOU = 0.60

# ============================================================
# DERIVED PATHS — NORMALLY DO NOT EDIT MANUALLY
# ============================================================

DRIVE_ROOT = DRIVE_BASE / PROJECT_NAME
DATASET_ZIP = DRIVE_ROOT / DATASET_ZIP_NAME
RUNS_DIR = DRIVE_ROOT / 'runs'
TESTS_DIR = DRIVE_ROOT / 'tests'
EXPORTS_DIR = DRIVE_ROOT / 'exports'
TEST_INPUTS_DIR = DRIVE_ROOT / 'test_inputs'
DFC_DIR = DRIVE_ROOT / 'hailo_dfc'

LOCAL_EXTRACT = Path('/content') / f'{PROJECT_SLUG}_dataset'
DATA_YAML = Path('/content') / f'{PROJECT_SLUG}_data.yaml'
DATA_YAML_FILENAME = f'{PROJECT_SLUG}_data.yaml'
RUN_PREFIX = f'{PROJECT_SLUG}_{Path(BASE_MODEL).stem}'
DOCKER_EXPORT_FOLDER = f'{PROJECT_SLUG}_export'

for folder in [DRIVE_ROOT, RUNS_DIR, TESTS_DIR, EXPORTS_DIR,
               TEST_INPUTS_DIR, DFC_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

assert TRAIN_MODE in {'new', 'resume', 'extend', 'finetune'}
assert PROJECT_SLUG and all(c.isalnum() or c in {'-', '_'} for c in PROJECT_SLUG), (
    'PROJECT_SLUG may contain only letters, numbers, hyphens, and underscores.'
)
assert CLASS_NAMES, 'CLASS_NAMES cannot be empty.'
assert len(CLASS_NAMES) == len(set(CLASS_NAMES)), 'CLASS_NAMES contains duplicate names.'
assert IMAGE_SIZE > 0, 'IMAGE_SIZE must be positive.'
assert DATASET_ZIP.exists(), f'Dataset not found: {DATASET_ZIP}'

print('Project:', PROJECT_NAME)
print('Dataset:', DATASET_ZIP)
print('Base model:', BASE_MODEL)
print('Training mode:', TRAIN_MODE)
print('Class count:', len(CLASS_NAMES))
print('Classes:', CLASS_NAMES)
print('Hailo network:', HAILO_MODEL_ZOO_NETWORK)
print('Hailo target:', HAILO_TARGET_DESCRIPTION)


In [ ]:
# Install Ultralytics and record the environment versions in Drive.
!pip -q install -U ultralytics pyyaml

import ultralytics, torch
print('Ultralytics:', ultralytics.__version__)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU is not enabled. Select a GPU in the Runtime settings.'

(DRIVE_ROOT / 'environment.txt').write_text(
    f'python={sys.version}\n'
    f'ultralytics={ultralytics.__version__}\n'
    f'torch={torch.__version__}\n',
    encoding='utf-8'
)

## 2. Extract the Dataset to Colab's Local Disk

The ZIP remains in Google Drive. During each new Colab session, it is extracted to `/content` for faster training and does not need to be downloaded from the internet again.


In [ ]:
import shutil, zipfile

if LOCAL_EXTRACT.exists():
    shutil.rmtree(LOCAL_EXTRACT)
LOCAL_EXTRACT.mkdir(parents=True)

with zipfile.ZipFile(DATASET_ZIP, 'r') as zf:
    zf.extractall(LOCAL_EXTRACT)

# Find the dataset root even if the ZIP contains one additional outer directory.
candidates = []
for p in [LOCAL_EXTRACT, *[x for x in LOCAL_EXTRACT.rglob('*') if x.is_dir()]]:
    if (p / 'images' / 'train').is_dir() and (p / 'labels' / 'train').is_dir():
        candidates.append(p)

if not candidates:
    raise FileNotFoundError(
        'The ZIP does not contain the required images/train and labels/train structure.'
    )

DATASET_ROOT = min(candidates, key=lambda p: len(p.parts))

def split_name(base):
    if (DATASET_ROOT / base / 'val').is_dir():
        return 'val'
    if (DATASET_ROOT / base / 'validation').is_dir():
        return 'validation'
    return None

VAL_NAME = split_name('images')
assert VAL_NAME is not None, 'Neither images/val nor images/validation was found.'
assert (DATASET_ROOT / 'labels' / VAL_NAME).is_dir(), 'Validation labels were not found.'
assert (DATASET_ROOT / 'images' / 'test').is_dir(), 'images/test was not found.'
assert (DATASET_ROOT / 'labels' / 'test').is_dir(), 'labels/test was not found.'

print('Dataset root:', DATASET_ROOT)


In [ ]:
# Generate data.yaml
import yaml

yaml_data = {
    'path': str(DATASET_ROOT),
    'train': 'images/train',
    'val': f'images/{VAL_NAME}',
    'test': 'images/test',
    'nc': len(CLASS_NAMES),
    'names': {i: name for i, name in enumerate(CLASS_NAMES)},
}
DATA_YAML.write_text(yaml.safe_dump(yaml_data, sort_keys=False), encoding='utf-8')
shutil.copy2(DATA_YAML, DRIVE_ROOT / DATA_YAML_FILENAME)
print(DATA_YAML.read_text())


## 3. Dataset Validation

This cell checks for missing annotations, invalid class IDs, and bounding-box values outside the `0–1` range. Do not start training if an error is reported.


In [ ]:
from collections import Counter

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
report = {}
total_errors = []

for split in ['train', VAL_NAME, 'test']:
    image_dir = DATASET_ROOT / 'images' / split
    label_dir = DATASET_ROOT / 'labels' / split
    images = sorted(p for p in image_dir.rglob('*') if p.suffix.lower() in IMG_EXTS)
    counts = Counter()
    missing = 0

    for image_path in images:
        relative = image_path.relative_to(image_dir).with_suffix('.txt')
        label_path = label_dir / relative
        if not label_path.exists():
            missing += 1
            total_errors.append(f'Missing label: {label_path}')
            continue

        for line_no, line in enumerate(label_path.read_text(encoding='utf-8').splitlines(), 1):
            if not line.strip():
                continue
            parts = line.split()
            if len(parts) != 5:
                total_errors.append(f'Invalid column count: {label_path}:{line_no}')
                continue
            try:
                class_id = int(parts[0])
                coords = [float(x) for x in parts[1:]]
            except ValueError:
                total_errors.append(f'Numeric parsing error: {label_path}:{line_no}')
                continue
            if not 0 <= class_id < len(CLASS_NAMES):
                total_errors.append(f'Invalid class ID: {label_path}:{line_no} -> {class_id}')
            if not all(0.0 <= value <= 1.0 for value in coords):
                total_errors.append(f'Coordinate outside the 0-1 range: {label_path}:{line_no}')
            counts[class_id] += 1

    report[split] = {'images': len(images), 'missing_labels': missing, 'boxes': counts}

for split, info in report.items():
    print(f'\n[{split}] images={info["images"]}, missing_labels={info["missing_labels"]}')
    for idx, name in enumerate(CLASS_NAMES):
        print(f'  {idx} {name:>6}: {info["boxes"][idx]} boxes')

if total_errors:
    print('\nFirst 30 errors:')
    print('\n'.join(total_errors[:30]))
    raise ValueError(f'Dataset validation found {len(total_errors)} errors.')

print('\nDataset structural validation completed successfully.')

## 4. Visually Inspect the Annotations

This cell displays bounding boxes and class names on random samples. Passing structural validation does not guarantee that boxes were drawn around the correct objects; always inspect these samples.


In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

random.seed(SEED)
train_images = [p for p in (DATASET_ROOT / 'images' / 'train').rglob('*')
                if p.suffix.lower() in IMG_EXTS]
sample_images = random.sample(train_images, min(VISUAL_SAMPLE_COUNT, len(train_images)))

cols = min(4, max(1, VISUAL_SAMPLE_COUNT))
rows = max(1, (VISUAL_SAMPLE_COUNT + cols - 1) // cols)
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows), squeeze=False)
for ax, image_path in zip(axes.flat, sample_images):
    image = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(image)
    w, h = image.size
    relative = image_path.relative_to(DATASET_ROOT / 'images' / 'train').with_suffix('.txt')
    label_path = DATASET_ROOT / 'labels' / 'train' / relative
    for line in label_path.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        class_id, xc, yc, bw, bh = map(float, line.split())
        class_id = int(class_id)
        x1, y1 = (xc - bw / 2) * w, (yc - bh / 2) * h
        x2, y2 = (xc + bw / 2) * w, (yc + bh / 2) * h
        draw.rectangle((x1, y1, x2, y2), outline='red', width=max(2, w // 400))
        draw.text((x1 + 3, y1 + 3), CLASS_NAMES[class_id], fill='yellow')
    ax.imshow(image)
    ax.set_title(image_path.name)
    ax.axis('off')

for ax in axes.flat[len(sample_images):]:
    ax.axis('off')
plt.tight_layout()

## 5. Model Training and Continuation

### New Training

In the configuration cell:

```python
TRAIN_MODE = 'new'
```

### Exact Resume After a Colab Interruption

```python
TRAIN_MODE = 'resume'
```

This mode restores the epoch, optimizer, and scheduler state stored in `last.pt`.

### Training Finished but the Results Are Not Sufficient

```python
TRAIN_MODE = 'extend'
EXTRA_EPOCHS = 40
```

This mode starts a new fine-tuning stage from the newest `best.pt` weights. It can be repeated after testing, but if validation or test performance worsens, the correct response is usually to improve the data or annotations rather than adding more epochs.


### Dataset Updated After Training

If you changed the main dataset after a previous training run — for example by **adding new images, removing images, or correcting annotations** — do not use `resume`. `resume` is for continuing the same interrupted training job with its previous optimizer/scheduler state.

Instead:

```python
TRAIN_MODE = 'finetune'
FINE_TUNE_EPOCHS = 30
FINE_TUNE_LR0 = 0.001
```

This mode loads the newest `best.pt` and trains it again on the **updated complete dataset**. It starts a new training run with a lower learning rate, so the existing model knowledge is retained while it adapts to the changed dataset.

Use this only when the class order/IDs remain compatible with the previous model. If you changed class meanings or class IDs, do not fine-tune blindly; train a new model after fixing the dataset.

Example:

```python
# 1. Replace the dataset ZIP with the updated version.
# 2. Keep the same CLASS_NAMES and class-ID order.
# 3. Run the dataset extraction and validation cells.
# 4. Then:
TRAIN_MODE = 'finetune'
FINE_TUNE_EPOCHS = 30
FINE_TUNE_LR0 = 0.001
```

**Important:** `finetune` is different from `extend`. `extend` adds more epochs because the dataset is essentially unchanged. `finetune` is specifically for retraining from the previous `best.pt` after the dataset itself has changed.


In [ ]:
from datetime import datetime
from ultralytics import YOLO

def newest_checkpoint(filename):
    files = list(RUNS_DIR.rglob(filename))
    if not files:
        raise FileNotFoundError(f'{filename} not found: {RUNS_DIR}')
    return max(files, key=lambda p: p.stat().st_mtime)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

if TRAIN_MODE == 'new':
    model = YOLO(BASE_MODEL)
    run_name = f'{RUN_PREFIX}_{timestamp}'
    train_results = model.train(
        data=str(DATA_YAML),
        epochs=NEW_EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH,
        patience=PATIENCE,
        close_mosaic=CLOSE_MOSAIC,
        workers=WORKERS,
        seed=SEED,
        pretrained=True,
        project=str(RUNS_DIR),
        name=run_name,
        plots=True,
        save=True,
    )

elif TRAIN_MODE == 'resume':
    checkpoint = newest_checkpoint('last.pt')
    print('Exact-resume checkpoint:', checkpoint)
    model = YOLO(str(checkpoint))
    train_results = model.train(resume=True)

elif TRAIN_MODE == 'extend':
    checkpoint = newest_checkpoint('best.pt')
    print('Additional-training starting checkpoint:', checkpoint)
    model = YOLO(str(checkpoint))
    run_name = f'{RUN_PREFIX}_extend_{timestamp}'
    train_results = model.train(
        data=str(DATA_YAML),
        epochs=EXTRA_EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH,
        patience=min(PATIENCE, EXTRA_EPOCHS),
        close_mosaic=max(5, min(CLOSE_MOSAIC, EXTRA_EPOCHS // 4)),
        workers=WORKERS,
        seed=SEED,
        project=str(RUNS_DIR),
        name=run_name,
        plots=True,
        save=True,
    )


elif TRAIN_MODE == 'finetune':
    checkpoint = newest_checkpoint('best.pt')
    print('Fine-tuning from updated-dataset checkpoint:', checkpoint)
    model = YOLO(str(checkpoint))
    run_name = f'{RUN_PREFIX}_finetune_{timestamp}'
    train_results = model.train(
        data=str(DATA_YAML),
        epochs=FINE_TUNE_EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH,
        lr0=FINE_TUNE_LR0,
        patience=min(PATIENCE, FINE_TUNE_EPOCHS),
        close_mosaic=max(5, min(CLOSE_MOSAIC, FINE_TUNE_EPOCHS // 4)),
        workers=WORKERS,
        seed=SEED,
        project=str(RUNS_DIR),
        name=run_name,
        plots=True,
        save=True,
    )

BEST_PT = newest_checkpoint('best.pt')
LAST_PT = newest_checkpoint('last.pt')
print('Newest best.pt:', BEST_PT)
print('Newest last.pt:', LAST_PT)

## 6. Genuine Evaluation on a Separate Test Set

Do not judge training only by overall mAP. Examine difficult or easily confused classes, nighttime images, and per-class recall values.

This cell evaluates the `test` split and saves outputs, including the confusion matrix, to Drive.


In [ ]:
BEST_PT = newest_checkpoint('best.pt')
test_model = YOLO(str(BEST_PT))
test_name = f'test_{datetime.now().strftime("%Y%m%d_%H%M%S")}'

metrics = test_model.val(
    data=str(DATA_YAML),
    split='test',
    imgsz=IMAGE_SIZE,
    batch=TEST_BATCH,
    conf=TEST_CONF,
    iou=TEST_IOU,
    plots=True,
    project=str(TESTS_DIR),
    name=test_name,
)

print('\nTest results')
print('mAP50-95:', float(metrics.box.map))
print('mAP50:', float(metrics.box.map50))
print('mAP75:', float(metrics.box.map75))
print('\nPer-class mAP50-95:')
for idx, value in enumerate(metrics.box.maps):
    print(f'{idx} {CLASS_NAMES[idx]:>6}: {float(value):.4f}')
print('Test outputs:', TESTS_DIR / test_name)

# Append the test summary to persistent history for comparison with later training runs.
import json
TEST_HISTORY = DRIVE_ROOT / 'test_history.jsonl'
history_record = {
    'timestamp': datetime.now().isoformat(timespec='seconds'),
    'checkpoint': str(BEST_PT),
    'test_name': test_name,
    'map50_95': float(metrics.box.map),
    'map50': float(metrics.box.map50),
    'map75': float(metrics.box.map75),
    'precision': float(metrics.box.mp),
    'recall': float(metrics.box.mr),
    'per_class_map50_95': {
        CLASS_NAMES[i]: float(value) for i, value in enumerate(metrics.box.maps)
    },
}
with TEST_HISTORY.open('a', encoding='utf-8') as f:
    f.write(json.dumps(history_record, ensure_ascii=False) + '\n')
print('Saved to test history:', TEST_HISTORY)

In [ ]:
# Display the confusion matrix and test curves
from IPython.display import display

test_output = TESTS_DIR / test_name
plot_files = [
    test_output / 'confusion_matrix_normalized.png',
    test_output / 'PR_curve.png',
    test_output / 'F1_curve.png',
]
for plot_file in plot_files:
    if plot_file.exists():
        print(plot_file.name)
        display(Image.open(plot_file))

## 6.1 Optional: Learning Curves, Previous-Test Comparison, and Saturation Analysis

Run this cell only when you need it. It produces:

- Combined mAP and loss curves for all `new`, `extend`, and `finetune` training runs.
- The change between the previous and latest test.
- An overfitting indicator.
- A **saturation indicator** showing whether the learning curve is approaching a plateau.

> A saturation value of 90% does not mean that the model has learned 90% of reality. It is only a heuristic indicating that the learning curve may be flattening with the current dataset and settings, and that additional epochs may provide limited benefit. New and higher-quality data can change the achievable limit.


In [ ]:
# OPTIONAL CELL
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

result_files = sorted(RUNS_DIR.rglob('results.csv'), key=lambda p: p.stat().st_mtime)
if not result_files:
    raise FileNotFoundError(f'results.csv not found: {RUNS_DIR}')

frames = []
global_start = 0
for result_file in result_files:
    frame = pd.read_csv(result_file)
    frame.columns = [column.strip() for column in frame.columns]
    frame['run'] = result_file.parent.name
    frame['global_epoch'] = np.arange(global_start + 1, global_start + len(frame) + 1)
    global_start += len(frame)
    frames.append(frame)

history = pd.concat(frames, ignore_index=True)

def find_column(candidates):
    for candidate in candidates:
        if candidate in history.columns:
            return candidate
    raise KeyError(f'Expected column not found: {candidates}')

map_col = find_column(['metrics/mAP50-95(B)', 'metrics/mAP50-95'])
map50_col = find_column(['metrics/mAP50(B)', 'metrics/mAP50'])
precision_col = find_column(['metrics/precision(B)', 'metrics/precision'])
recall_col = find_column(['metrics/recall(B)', 'metrics/recall'])
train_box_col = find_column(['train/box_loss'])
val_box_col = find_column(['val/box_loss'])

curve = history[map_col].astype(float).to_numpy()
epochs = history['global_epoch'].to_numpy()
smooth_window = min(7, max(1, len(curve) // 10))
smooth = pd.Series(curve).rolling(smooth_window, min_periods=1).mean().to_numpy()
recent_window = min(20, max(5, len(curve) // 4))

recent_x = np.arange(recent_window, dtype=float)
recent_y = smooth[-recent_window:]
recent_slope = float(np.polyfit(recent_x, recent_y, 1)[0]) if recent_window > 1 else 0.0
recent_gain = float(recent_y[-1] - recent_y[0])
best_value = float(np.max(smooth))
current_value = float(smooth[-1])
epochs_since_best = int(len(smooth) - 1 - int(np.argmax(smooth)))

# Use the early learning rate as a reference.
early_window = min(20, max(5, len(curve) // 4))
early_y = smooth[:early_window]
early_slope = float(np.polyfit(np.arange(early_window), early_y, 1)[0]) if early_window > 1 else 0.0
positive_reference = max(early_slope, 0.001)
slope_ratio = max(0.0, recent_slope) / positive_reference
plateau_from_slope = 1.0 - float(np.clip(slope_ratio, 0.0, 1.0))
plateau_from_age = 1.0 - math.exp(-epochs_since_best / 8.0)
plateau_from_gain = 1.0 - float(np.clip(max(recent_gain, 0.0) / 0.02, 0.0, 1.0))

train_loss = history[train_box_col].astype(float).to_numpy()
val_loss = history[val_box_col].astype(float).to_numpy()
loss_w = min(recent_window, len(train_loss))
train_loss_slope = float(np.polyfit(np.arange(loss_w), train_loss[-loss_w:], 1)[0])
val_loss_slope = float(np.polyfit(np.arange(loss_w), val_loss[-loss_w:], 1)[0])
map_drop = best_value - current_value
overfit = bool(train_loss_slope < 0 and val_loss_slope > 0 and map_drop > 0.01)

saturation = 100.0 * (
    0.50 * plateau_from_slope +
    0.25 * plateau_from_age +
    0.25 * plateau_from_gain
)
saturation = float(np.clip(saturation, 0.0, 100.0))

# Read persistent test history.
test_records = []
TEST_HISTORY = DRIVE_ROOT / 'test_history.jsonl'
if TEST_HISTORY.exists():
    for line in TEST_HISTORY.read_text(encoding='utf-8').splitlines():
        if line.strip():
            test_records.append(json.loads(line))

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

axes[0].plot(epochs, history[map50_col], alpha=0.35, label='mAP50')
axes[0].plot(epochs, curve, alpha=0.35, label='mAP50-95 raw')
axes[0].plot(epochs, smooth, linewidth=2.5, label='mAP50-95 smoothed')
axes[0].axhline(best_value, color='green', linestyle='--', alpha=0.6,
                label=f'Best {best_value:.3f}')
axes[0].set_title(f'Learning curve — saturation ≈ %{saturation:.0f}')
axes[0].set_xlabel('Combined epoch')
axes[0].set_ylabel('Metric')
axes[0].grid(alpha=0.25)
axes[0].legend(fontsize=8)

axes[1].plot(epochs, train_loss, label='Train box loss')
axes[1].plot(epochs, val_loss, label='Validation box loss')
axes[1].set_title('Train / validation loss')
axes[1].set_xlabel('Combined epoch')
axes[1].grid(alpha=0.25)
axes[1].legend()

if test_records:
    test_x = np.arange(1, len(test_records) + 1)
    for key, label in [
        ('map50_95', 'Test mAP50-95'),
        ('map50', 'Test mAP50'),
        ('precision', 'Test precision'),
        ('recall', 'Test recall'),
    ]:
        axes[2].plot(test_x, [r[key] for r in test_records], marker='o', label=label)
    axes[2].set_xticks(test_x)
    axes[2].set_xlabel('Test run order')
    axes[2].set_title('Comparison with previous tests')
    axes[2].grid(alpha=0.25)
    axes[2].legend(fontsize=8)
else:
    axes[2].text(0.5, 0.5, 'No test history yet', ha='center', va='center')
    axes[2].set_axis_off()

plt.suptitle(
    f'Last {recent_window} epochs gain: {recent_gain:+.4f} | '
    f'Slope: {recent_slope:+.6f}/epoch | Difference from best: {map_drop:.4f}',
    fontsize=11
)
plt.tight_layout()

analysis_dir = TESTS_DIR / f'learning_analysis_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
analysis_dir.mkdir(parents=True, exist_ok=True)
graph_path = analysis_dir / 'learning_and_test_comparison.png'
fig.savefig(graph_path, dpi=160, bbox_inches='tight')
plt.show()

print(f'\nSaturation/plateau proximity indicator: %{saturation:.1f}')
print(f'Last {recent_window} epochs mAP50-95 change: {recent_gain:+.4f}')
print(f'Best smoothed mAP50-95: {best_value:.4f}')
print(f'Current smoothed mAP50-95: {current_value:.4f}')
print(f'Epochs since the best value: {epochs_since_best}')
print('Overfitting indicator:', 'DETECTED' if overfit else 'not evident')

if len(test_records) >= 2:
    previous, latest = test_records[-2], test_records[-1]
    print('\nLatest change compared with the previous test:')
    for key in ['map50_95', 'map50', 'precision', 'recall']:
        delta = latest[key] - previous[key]
        print(f'  {key:>10}: {previous[key]:.4f} → {latest[key]:.4f} ({delta:+.4f})')

if overfit:
    decision = 'MORE EPOCHS ARE NOT RECOMMENDED: validation loss is increasing and mAP has fallen from its best value.'
elif saturation >= 85 and recent_gain < 0.005:
    decision = 'VERY CLOSE TO A PLATEAU: more epochs will probably provide little benefit; add new and diverse data.'
elif recent_gain >= 0.01 and recent_slope > 0:
    decision = 'THE MODEL IS STILL LEARNING: a controlled 20–40 epoch extension can be tested.'
else:
    decision = 'BORDERLINE/UNCERTAIN: run a controlled 20-epoch extension and compare the change on the independent test set.'

print('\nDecision:', decision)
print('Graph saved:', graph_path)

summary = {
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'saturation_indicator_percent': saturation,
    'recent_window_epochs': recent_window,
    'recent_map50_95_gain': recent_gain,
    'recent_map50_95_slope_per_epoch': recent_slope,
    'best_smoothed_map50_95': best_value,
    'current_smoothed_map50_95': current_value,
    'epochs_since_best': epochs_since_best,
    'overfit_signal': overfit,
    'decision': decision,
    'warning': 'The saturation indicator is not an exact percentage of true model learning capacity.',
}
(analysis_dir / 'analysis_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
)

## 7. Visual Testing with Your Own Images

Place new images that you want to test in the `TEST_INPUTS_DIR` directory derived in the user-configuration cell.

Default example:

```text
MyDrive/<PROJECT_NAME>/test_inputs/
```

These images must not be part of the training dataset. The cell saves predictions with bounding boxes under `tests/predictions_*` and displays the first samples.


In [ ]:
BEST_PT = newest_checkpoint('best.pt')
predict_model = YOLO(str(BEST_PT))
predict_name = f'predictions_{datetime.now().strftime("%Y%m%d_%H%M%S")}'

input_images = [p for p in TEST_INPUTS_DIR.rglob('*') if p.suffix.lower() in IMG_EXTS]
if not input_images:
    raise FileNotFoundError(f'No test image found: {TEST_INPUTS_DIR}')

results = predict_model.predict(
    source=str(TEST_INPUTS_DIR),
    imgsz=IMAGE_SIZE,
    conf=PREDICT_CONF,
    iou=PREDICT_IOU,
    save=True,
    project=str(TESTS_DIR),
    name=predict_name,
)

predict_output = TESTS_DIR / predict_name
saved_images = [p for p in predict_output.rglob('*') if p.suffix.lower() in IMG_EXTS]
print('Predictions:', predict_output)

for image_path in saved_images[:MAX_PREDICTION_PREVIEWS]:
    display(Image.open(image_path))

## 8. PT and ONNX Export

The selected `best.pt` is exported to ONNX and copied to Drive before Hailo compilation.

The ONNX input size, opset, simplification, dynamic-shape setting, and batch size are controlled by the initial user-configuration cell. Hailo generally requires a fixed input size, `dynamic=False`, and `batch=1`.


In [ ]:
BEST_PT = newest_checkpoint('best.pt')
export_stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
EXPORT_DIR = EXPORTS_DIR / f'{PROJECT_SLUG}_{export_stamp}'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(BEST_PT, EXPORT_DIR / 'best.pt')
shutil.copy2(DATA_YAML, EXPORT_DIR / DATA_YAML_FILENAME)

export_model = YOLO(str(BEST_PT))
onnx_path = Path(export_model.export(
    format='onnx',
    imgsz=IMAGE_SIZE,
    opset=ONNX_OPSET,
    simplify=ONNX_SIMPLIFY,
    dynamic=ONNX_DYNAMIC,
    batch=ONNX_BATCH,
))
shutil.copy2(onnx_path, EXPORT_DIR / 'best.onnx')
print('PT:', EXPORT_DIR / 'best.pt')
print('ONNX:', EXPORT_DIR / 'best.onnx')

## 9A. Direct HEF Export — Preferred Method

Hailo HEF compilation requires the Hailo Dataflow Compiler. Standard Colab environments do not include it.

1. Download a DFC `.whl` from the Hailo Developer Zone that is compatible with the target hardware, Linux, and the Colab Python version.
2. Place it once in `MyDrive/<PROJECT_NAME>/hailo_dfc/`.
3. Run the installation cell below.

This section cannot run without a compatible wheel. Running the Hailo AI Software Suite Docker archive inside Colab is not reliable; in that case, compile the **9B** package in a normal Ubuntu x86-64 Hailo Docker environment.

Verify that `HAILO_MODEL_ZOO_NETWORK` matches the trained YOLO architecture. Changing the model size involves more than changing `BASE_MODEL`; the Hailo parser YAML network must use the same architecture.


In [ ]:
# Install the Hailo DFC wheel. Use method 9B if this fails.
import glob

dfc_wheels = sorted(DFC_DIR.glob('hailo_dataflow_compiler*.whl'))
if not dfc_wheels:
    raise FileNotFoundError(
        f'No compatible Hailo DFC wheel found: {DFC_DIR}'
    )

DFC_WHEEL = dfc_wheels[-1]
print('DFC to install:', DFC_WHEEL.name)
subprocess.run([sys.executable, '-m', 'pip', 'install', str(DFC_WHEEL)], check=True)
print('DFC installed. If an import error occurs, restart the runtime and run Sections 0, 1, 8, and 9A again.')

In [ ]:
# Direct YOLO → Hailo INT8 HEF export
# HAILO_CONF may be written into the HEF NMS configuration; the application may apply a higher threshold.
BEST_PT = newest_checkpoint('best.pt')
hef_model = YOLO(str(BEST_PT))

hef_output = Path(hef_model.export(
    format='hailo',
    name=HAILO_EXPORT_NAME,
    imgsz=IMAGE_SIZE,
    data=str(DATA_YAML),
    fraction=HAILO_CALIBRATION_FRACTION,
    conf=HAILO_CONF,
    iou=HAILO_IOU,
))

# Export may return a directory or a file; copy the complete output to Drive.
direct_dir = EXPORT_DIR / f'{HAILO_EXPORT_NAME}_direct'
if direct_dir.exists():
    shutil.rmtree(direct_dir)
if hef_output.is_dir():
    shutil.copytree(hef_output, direct_dir)
else:
    direct_dir.mkdir(parents=True)
    shutil.copy2(hef_output, direct_dir / hef_output.name)

print('Hailo export:', direct_dir)
print('Contents:', [p.name for p in direct_dir.iterdir()])


## 9B. Docker Compilation Package When DFC Does Not Work in Colab

This cell creates a package for use in an Ubuntu x86-64 Hailo AI Software Suite Docker environment:

- `best.pt`
- `best.onnx`
- A project-specific `data.yaml`
- Representative images for Hailo INT8 calibration
- A `compile_hailo.sh` generated automatically from the project, class count, model network, and hardware settings
- `README_HAILO.txt` describing the package contents

Calibration images do not require label files. However, the selected images should represent the real deployment distribution as closely as possible.

> The generated script uses the configured model network and class count, but you must still verify that the corresponding YAML file exists in the installed Hailo Model Zoo version.


In [ ]:
# Copy representative calibration images from train/val.
calibration_dir = EXPORT_DIR / 'calibration_images'
if calibration_dir.exists():
    shutil.rmtree(calibration_dir)
calibration_dir.mkdir(parents=True)

all_calib_candidates = []
for split in ['train', VAL_NAME]:
    all_calib_candidates.extend(
        p for p in (DATASET_ROOT / 'images' / split).rglob('*')
        if p.suffix.lower() in IMG_EXTS
    )

random.Random(SEED).shuffle(all_calib_candidates)
selected = all_calib_candidates[:min(HAILO_CALIBRATION_COUNT, len(all_calib_candidates))]
for idx, src in enumerate(selected):
    shutil.copy2(src, calibration_dir / f'{idx:05d}_{src.name}')

model_zoo_yaml = (
    '/local/workspace/hailo_model_zoo/hailo_model_zoo/cfg/networks/'
    f'{HAILO_MODEL_ZOO_NETWORK}.yaml'
)

compile_script = '\n'.join([
    '#!/usr/bin/env bash',
    'set -euo pipefail',
    '',
    '# Example usage inside Hailo AI Software Suite Docker.',
    '# The shared directory and Model Zoo path may differ depending on the installation.',
    f'MODEL_ZOO_YAML={model_zoo_yaml}',
    '',
    'hailomz compile \\',
    f'  --ckpt /local/shared_with_docker/{DOCKER_EXPORT_FOLDER}/best.onnx \\',
    f'  --calib-path /local/shared_with_docker/{DOCKER_EXPORT_FOLDER}/calibration_images \\',
    '  --yaml "$MODEL_ZOO_YAML" \\',
    f'  --classes {len(CLASS_NAMES)} \\',
    f'  --hw-arch {HAILO_HW_ARCH}',
    '',
])
(EXPORT_DIR / 'compile_hailo.sh').write_text(compile_script, encoding='utf-8')

readme = '\n'.join([
    f'{PROJECT_NAME} Hailo export package',
    '',
    f'Base model: {BASE_MODEL}',
    f'Hailo Model Zoo network: {HAILO_MODEL_ZOO_NETWORK}',
    f'Class count: {len(CLASS_NAMES)}',
    f'Classes: {", ".join(CLASS_NAMES)}',
    f'Input: {IMAGE_SIZE}x{IMAGE_SIZE}, batch {ONNX_BATCH}',
    f'Target: {HAILO_TARGET_DESCRIPTION} ({HAILO_HW_ARCH})',
    '',
    f'1. Place this directory under the Docker host shared_with_docker directory using the name {DOCKER_EXPORT_FOLDER}.',
    f'2. Verify that {HAILO_MODEL_ZOO_NETWORK}.yaml exists in Hailo Model Zoo.',
    '3. Check the paths in compile_hailo.sh for your installation.',
    '4. Run the script inside the Hailo Docker environment.',
    '',
    'Important: the ONNX model architecture must match the Hailo parser/YAML architecture.',
    '',
])
(EXPORT_DIR / 'README_HAILO.txt').write_text(readme, encoding='utf-8')

archive = shutil.make_archive(str(EXPORT_DIR) + '_docker_package', 'zip', EXPORT_DIR)
print('Calibration images:', len(selected))
print('Docker package:', archive)


## 10. Validate the HEF on the Target Device

After copying the generated `.hef` file to the target device, validate it using its actual filename:

```bash
hailortcli fw-control identify
hailortcli parse-hef MODEL.hef
hailortcli benchmark MODEL.hef
```

Do not make the final decision using only `.pt` test results. Evaluate the `.hef` on the same field images. After INT8 quantization, pay particular attention to classes that are visually similar and commonly confused.

### Correct Iteration Loop

```text
new training
   ↓
test split + new field images
   ↓
If the error is caused by annotations, fix the dataset
   ↓
If training was interrupted, use resume
If training finished but more data was added, use extend
   ↓
PT test
   ↓
ONNX / HEF export
   ↓
HEF test
```

More epochs are not always better. Extending training while test performance is declining may increase overfitting; in that situation, adding more diverse real-world data is often the better solution.
